In [0]:
import requests
import json
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql.functions import col, max as spark_max
from delta.tables import DeltaTable
import hashlib

# Medallion Architecture: Bronze → Silver → Gold

This notebook implements a full **medallion architecture** using the free **JSONPlaceholder API** (https://jsonplaceholder.typicode.com):

| Layer | Purpose | Description |
| --- | --- | --- |
| **Bronze** | Raw ingestion | Fetch raw JSON from API endpoints (users, posts, comments) and store as Delta tables |
| **Silver** | Cleansed & conformed | Flatten JSON, cast types, remove duplicates, add metadata columns |
| **Gold** | Analytics-ready | Aggregated business tables: posts per user, comments per post, user activity summary |

**Source API:** JSONPlaceholder — a free fake REST API with no authentication required.

In [0]:


# ── Configuration ──────────────────────────────────────────────
CATALOG   = "main"
BASE_URL  = "https://jsonplaceholder.typicode.com"

# Define the three medallion schemas
SCHEMAS = {
    "bronze": f"{CATALOG}.bronze_jsonplaceholder",
    "silver": f"{CATALOG}.silver_jsonplaceholder",
    "gold":   f"{CATALOG}.gold_jsonplaceholder",
}

# API endpoints to ingest
ENDPOINTS = ["users", "posts", "comments","albums","photos"]

# ── Create Catalog & Schemas ──────────────────────────────────
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
for schema_name in SCHEMAS.values():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

print(f"✅ Catalog '{CATALOG}' and schemas created:")
for layer, schema in SCHEMAS.items():
    print(f"   • {layer}: {schema}")

✅ Catalog 'main' and schemas created:
   • bronze: main.bronze_jsonplaceholder
   • silver: main.silver_jsonplaceholder
   • gold: main.gold_jsonplaceholder


## 🔵 Bronze Layer — Raw Ingestion

Fetch raw JSON from each API endpoint and persist it **as-is** into Delta tables. The bronze layer preserves the original payload with minimal transformation — only adding ingestion metadata (`ingested_at`, `source_endpoint`).

### Data Quality Checks at Ingestion

Each API call is validated and audited:

| Check | Column | Description |
| --- | --- | --- |
| **HTTP status** | `http_status_code` | Must be 200; otherwise the endpoint is marked as failed |
| **Response latency** | `response_latency_ms` | Round-trip time from request to response (ms) |
| **Payload record count** | `payload_record_count` | Number of records returned by the endpoint |
| **Payload size** | `payload_size_bytes` | Raw response body size in bytes |
| **Ingestion status** | `ingestion_status` | `success` or `failed` |
| **Error message** | `error_message` | Exception or non-200 status details (null on success) |

An **ingestion audit log** (`main.bronze_jsonplaceholder.ingestion_audit_log`) records every endpoint attempt — including failures — for full auditability.

In [0]:
# for endpoint in ENDPOINTS:
#     print(f"   📥 Fetching {endpoint} from API...")
#     response = requests.get(f"{BASE_URL}/{endpoint}", timeout=30)
#     print(response.json())


In [0]:
# for endpoint in ENDPOINTS:

#     print(f"📥 Fetching {endpoint} from API...")

#     table_name = f"{SCHEMAS['bronze']}.raw_{endpoint}"

    
#     spark.sql(f"""DROP TABLE IF EXISTS {table_name}""")

    # spark.sql(f"""
    #     ALTER TABLE {table_name}
    #     ADD COLUMNS (
    #         record_id STRING,
    #         record_hash STRING
    #     )
    # """)

In [0]:



# ============================================================
# 1. INGESTION TIMESTAMP
# ============================================================

INGESTION_TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


# ============================================================
# 2. BRONZE SCHEMA
# ============================================================

bronze_schema = StructType([
    StructField("raw_payload",          StringType()),
    StructField("record_id",            StringType()),
    StructField("record_hash",          StringType()),
    StructField("ingested_at",          StringType()),
    StructField("source_endpoint",      StringType()),
    StructField("http_status_code",     IntegerType()),
    StructField("response_latency_ms",  IntegerType()),
    StructField("payload_record_count", IntegerType()),
    StructField("payload_size_bytes",   LongType()),
    StructField("ingestion_status",     StringType()),
    StructField("error_message",        StringType()),
])


# ============================================================
# 3. AUDIT SCHEMA
# ============================================================

audit_schema = StructType([
    StructField("endpoint",             StringType()),
    StructField("ingested_at",          StringType()),
    StructField("http_status_code",     IntegerType()),
    StructField("response_latency_ms",  IntegerType()),
    StructField("payload_record_count", IntegerType()),
    StructField("payload_size_bytes",   LongType()),
    StructField("ingestion_status",     StringType()),
    StructField("error_message",        StringType()),
])


audit_rows = []


# ============================================================
# 4. PROCESS EACH ENDPOINT
# ============================================================

for endpoint in ENDPOINTS:

    print(f"📥 Fetching {endpoint} from API...")

    table_name = f"{SCHEMAS['bronze']}.raw_{endpoint}"

    start_time = datetime.utcnow()

    http_status = None
    latency_ms = 0
    num_records = 0
    payload_bytes = 0
    status = "failed"
    error_msg = None


    try:

        # ====================================================
        # 5. API CALL
        # ====================================================

        response = requests.get(
            f"{BASE_URL}/{endpoint}",
            timeout=30
        )

        latency_ms = int(
            (datetime.utcnow() - start_time).total_seconds() * 1000
        )

        http_status = response.status_code


        # ====================================================
        # 6. SUCCESSFUL RESPONSE
        # ====================================================

        if http_status == 200:

            raw_data = response.json()

            num_records = len(raw_data)

            payload_bytes = len(response.content)

            status = "success"


            # =================================================
            # 7. CREATE BRONZE RECORDS
            # =================================================

            rows = []

            for record in raw_data:

                # ------------------------------------------------
                # Record ID
                # ------------------------------------------------
                #
                # Every JSONPlaceholder endpoint has an "id"
                # except that relationships use fields such as
                # userId/postId/albumId.
                #
                # The "id" uniquely identifies the record
                # within that endpoint.
                # ------------------------------------------------

                record_id = str(record.get("id"))


                # ------------------------------------------------
                # Create deterministic JSON
                # ------------------------------------------------
                #
                # sort_keys=True ensures that the same record
                # produces the same JSON representation.
                # ------------------------------------------------

                normalized_record = json.dumps(
                    record,
                    sort_keys=True,
                    separators=(",", ":")
                )


                # ------------------------------------------------
                # Create SHA-256 hash
                # ------------------------------------------------

                record_hash = __import__("hashlib").sha256(
                    normalized_record.encode("utf-8")
                ).hexdigest()


                # ------------------------------------------------
                # Add row
                # ------------------------------------------------

                rows.append(
                    (
                        normalized_record,
                        record_id,
                        record_hash,
                        INGESTION_TS,
                        endpoint,
                        http_status,
                        latency_ms,
                        num_records,
                        payload_bytes,
                        status,
                        None
                    )
                )


            # ====================================================
            # 8. CREATE SPARK DATAFRAME
            # ====================================================

            raw_df = spark.createDataFrame(
                rows,
                bronze_schema
            )


            # ====================================================
            # 9. CHECK WHETHER BRONZE TABLE EXISTS
            # ====================================================

            table_name_exists = spark.catalog.tableExists(
                table_name
            )

          


            # ====================================================
            # 10. INITIAL LOAD
            # ====================================================

            if not table_name_exists:

                print(
                    f"   📥 Creating table {table_name}"
                )


                (
                    raw_df
                    .write
                    .format("delta")
                    .mode("overwrite")
                    .saveAsTable(table_name)
                )


                insert_record_count = raw_df.count()


                print(
                    f"   ✅ Initial Load: "
                    f"{insert_record_count} records inserted "
                    f"into {table_name}"
                )


            # ====================================================
            # 11. INCREMENTAL LOAD
            # ====================================================

            else:

                print(
                    f"   📥 Checking for new/changed records "
                    f"in {table_name}"
                )


                # ------------------------------------------------
                # Read existing Bronze table
                # ------------------------------------------------

                target_df = spark.table(table_name)


                # ------------------------------------------------
                # Compare source with target
                #
                # LEFT JOIN:
                #
                # source.record_id = target.record_id
                #
                # This tells us whether the record already exists.
                # ------------------------------------------------

                # print("RAW DF COLUMNS:")
                # print(raw_df.columns)

                # print("TARGET DF COLUMNS:")
                # print(target_df.columns)

                # raw_df.printSchema()
                # target_df.printSchema()


                comparison_df = (
                    raw_df.alias("source")
                    .join(
                        target_df.select(
                            "record_id",
                            "record_hash"
                        ).alias("target"),
                        col("source.record_id")
                        == col("target.record_id"),
                        "left"
                    )
                )


                # ------------------------------------------------
                # Find:
                #
                # 1. New records
                # 2. Changed records
                #
                # New:
                # target.record_id IS NULL
                #
                # Changed:
                # source.record_hash != target.record_hash
                # ------------------------------------------------

                incremental_df = (
                    comparison_df
                    .filter(
                        col("target.record_id").isNull()
                        |
                        (
                            col("source.record_hash")
                            !=
                            col("target.record_hash")
                        )
                    )
                    .select("source.*")
                )


                # =================================================
                # COUNT CHANGED / NEW RECORDS
                # =================================================

                incremental_count = incremental_df.count()


                print(
                    f"   📊 New/changed records: "
                    f"{incremental_count}"
                )


                # =================================================
                # 12. MERGE
                # =================================================

                if incremental_count > 0:

                    target = DeltaTable.forName(
                        spark,
                        table_name
                    )


                    (
                        target.alias("target")
                        .merge(
                            incremental_df.alias("source"),

                            "target.record_id = source.record_id"
                        )

                        # Existing record whose hash changed
                        # → UPDATE ALL columns
                        .whenMatchedUpdateAll()

                        # New record
                        # → INSERT ALL columns
                        .whenNotMatchedInsertAll()

                        .execute()
                    )


                    print(
                        f"   ✅ MERGE completed"
                    )


                else:

                    print(
                        f"   ℹ️ No new/changed records"
                    )


            # ====================================================
            # 13. FINAL ROW COUNT
            # ====================================================

            row_count = (
                spark.table(table_name)
                .count()
            )


            print(
                f"   📦 Bronze table: {table_name}"
            )

            print(
                f"   📊 Total rows: {row_count}"
            )


        # ========================================================
        # 14. API FAILURE
        # ========================================================

        else:

            error_msg = (
                f"HTTP {http_status} — expected 200"
            )

            print(
                f"   ❌ {error_msg}"
            )


    # ============================================================
    # 15. REQUEST EXCEPTION
    # ============================================================

    except requests.exceptions.RequestException as ex:

        latency_ms = int(
            (datetime.utcnow() - start_time).total_seconds() * 1000
        )

        error_msg = str(ex)

        print(
            f"   ❌ Request exception: {error_msg}"
        )


    # ============================================================
    # 16. AUDIT RECORD
    # ============================================================

    audit_rows.append(
        (
            endpoint,
            INGESTION_TS,
            http_status,
            latency_ms,
            num_records,
            payload_bytes,
            status,
            error_msg
        )
    )


# ================================================================
# 17. WRITE AUDIT LOG
# ================================================================

audit_df = spark.createDataFrame(
    audit_rows,
    audit_schema
)


(
    audit_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        f"{SCHEMAS['bronze']}.ingestion_audit_log"
    )
)


# ================================================================
# 18. AUDIT SUMMARY
# ================================================================

failed = [
    r for r in audit_rows
    if r[6] != "success"
]


print(
    f"📊 Ingestion audit log: "
    f"{len(audit_rows)} endpoints | "
    f"{len(audit_rows) - len(failed)} succeeded | "
    f"{len(failed)} failed"
)


print(
    f"   → {SCHEMAS['bronze']}.ingestion_audit_log"
)


print(
    "🎉 Bronze layer complete!"
)


/home/spark-c23eba55-7041-4e7a-a511-fc/.ipykernel/66/command-6166947983077695-1166700394:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  INGESTION_TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
/home/spark-c23eba55-7041-4e7a-a511-fc/.ipykernel/66/command-6166947983077695-1166700394:59: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time = datetime.utcnow()
/home/spark-c23eba55-7041-4e7a-a511-fc/.ipykernel/66/command-6166947983077695-1166700394:81: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (datetime.utc

📥 Fetching users from API...
   📥 Checking for new/changed records in main.bronze_jsonplaceholder.raw_users
   📊 New/changed records: 0
   ℹ️ No new/changed records
   📦 Bronze table: main.bronze_jsonplaceholder.raw_users
   📊 Total rows: 10
📥 Fetching posts from API...
   📥 Checking for new/changed records in main.bronze_jsonplaceholder.raw_posts
   📊 New/changed records: 0
   ℹ️ No new/changed records
   📦 Bronze table: main.bronze_jsonplaceholder.raw_posts
   📊 Total rows: 100
📥 Fetching comments from API...
   📥 Checking for new/changed records in main.bronze_jsonplaceholder.raw_comments
   📊 New/changed records: 0
   ℹ️ No new/changed records
   📦 Bronze table: main.bronze_jsonplaceholder.raw_comments
   📊 Total rows: 500
📥 Fetching albums from API...
   📥 Checking for new/changed records in main.bronze_jsonplaceholder.raw_albums
   📊 New/changed records: 0
   ℹ️ No new/changed records
   📦 Bronze table: main.bronze_jsonplaceholder.raw_albums
   📊 Total rows: 100
📥 Fetching photos

In [0]:
%sql 
select * from main.bronze_jsonplaceholder.raw_users

raw_payload,record_id,record_hash,ingested_at,source_endpoint,http_status_code,response_latency_ms,payload_record_count,payload_size_bytes,ingestion_status,error_message
"{""address"":{""city"":""Gwenborough"",""geo"":{""lat"":""-37.3159"",""lng"":""81.1496""},""street"":""Kulas Light"",""suite"":""Apt. 556"",""zipcode"":""92998-3874""},""company"":{""bs"":""harness real-time e-markets"",""catchPhrase"":""Multi-layered client-server neural-net"",""name"":""Romaguera-Crona""},""email"":""Sincere@april.biz"",""id"":1,""name"":""Leanne Graham"",""phone"":""1-770-736-8031 x56442"",""username"":""Bret"",""website"":""hildegard.org""}",1,5ec7ec7fb081d215e28649e7d467b3d0f21f3638a28efb90db54cb472b10e09a,2026-10-08 08:05:25,users,200,61,10,5645,success,null
"{""address"":{""city"":""Wisokyburgh"",""geo"":{""lat"":""-43.9509"",""lng"":""-34.4618""},""street"":""Victor Plains"",""suite"":""Suite 879"",""zipcode"":""90566-7771""},""company"":{""bs"":""synergize scalable supply-chains"",""catchPhrase"":""Proactive didactic contingency"",""name"":""Deckow-Crist""},""email"":""Shanna@melissa.tv"",""id"":2,""name"":""Ervin Howell"",""phone"":""010-692-6593 x09125"",""username"":""Antonette"",""website"":""anastasia.net""}",2,0b58fdf0e343e2fc12abb8ad9c7416ea7422084b8541b74a3dedfc8f8c63cdcd,2026-10-08 08:05:25,users,200,61,10,5645,success,null
"{""address"":{""city"":""McKenziehaven"",""geo"":{""lat"":""-68.6102"",""lng"":""-47.0653""},""street"":""Douglas Extension"",""suite"":""Suite 847"",""zipcode"":""59590-4157""},""company"":{""bs"":""e-enable strategic applications"",""catchPhrase"":""Face to face bifurcated interface"",""name"":""Romaguera-Jacobson""},""email"":""Nathan@yesenia.net"",""id"":3,""name"":""Clementine Bauch"",""phone"":""1-463-123-4447"",""username"":""Samantha"",""website"":""ramiro.info""}",3,70525497a63eb477e01b10b2baaf0b3c638c19bc12f8624fd4ce69c284a2940b,2026-10-08 08:05:25,users,200,61,10,5645,success,null
"{""address"":{""city"":""South Elvis"",""geo"":{""lat"":""29.4572"",""lng"":""-164.2990""},""street"":""Hoeger Mall"",""suite"":""Apt. 692"",""zipcode"":""53919-4257""},""company"":{""bs"":""transition cutting-edge web services"",""catchPhrase"":""Multi-tiered zero tolerance productivity"",""name"":""Robel-Corkery""},""email"":""Julianne.OConner@kory.org"",""id"":4,""name"":""Patricia Lebsack"",""phone"":""493-170-9623 x156"",""username"":""Karianne"",""website"":""kale.biz""}",4,f34275e224bcbc2b4c41d3ed5f30c463646739502027f08c8cff1e8b1a5edcd6,2026-10-08 08:05:25,users,200,61,10,5645,success,null
"{""address"":{""city"":""Roscoeview"",""geo"":{""lat"":""-31.8129"",""lng"":""62.5342""},""street"":""Skiles Walks"",""suite"":""Suite 351"",""zipcode"":""33263""},""company"":{""bs"":""revolutionize end-to-end systems"",""catchPhrase"":""User-centric fault-tolerant solution"",""name"":""Keebler LLC""},""email"":""Lucio_Hettinger@annie.ca"",""id"":5,""name"":""Chelsey Dietrich"",""phone"":""(254)954-1289"",""username"":""Kamren"",""website"":""demarco.info""}",5,ea9b8c7a4bdfec880166c28e999b9ebce16feafc85173cd4bcc4bc69b2a1c601,2026-10-08 08:05:25,users,200,61,10,5645,success,null
"{""address"":{""city"":""South Christy"",""geo"":{""lat"":""-71.4197"",""lng"":""71.7478""},""street"":""Norberto Crossing"",""suite"":""Apt. 950"",""zipcode"":""23505-1337""},""company"":{""bs"":""e-enable innovative applications"",""catchPhrase"":""Synchronised bottom-line interface"",""name"":""Considine-Lockman""},""email"":""Karley_Dach@jasper.info"",""id"":6,""name"":""Mrs. Dennis Schulist"",""phone"":""1-477-935-8478 x6430"",""username"":""Leopoldo_Corkery"",""website"":""ola.org""}",6,0ba39e2b6a4a924c17eca2b1cba045329166821e44f1e56f5d6568d5bd69d250,2026-10-08 08:05:25,users,200,61,10,5645,success,null
"{""address"":{""city"":""Howemouth"",""geo"":{""lat"":""24.8918"",""lng"":""21.8984""},""street"":""Rex Trail"",""suite"":""Suite 280"",""zipcode"":""58804-1099""},""company"":{""bs"":""generate enterprise e-tailers"",""catchPhrase"":""Confi

In [0]:
%sql
select * from main.bronze_jsonplaceholder.raw_albums

raw_payload,record_id,record_hash,ingested_at,source_endpoint,http_status_code,response_latency_ms,payload_record_count,payload_size_bytes,ingestion_status,error_message
"{""id"":1,""title"":""quidem molestiae enim"",""userId"":1}",1,dae6864bc4b2227e446fbd13223377aab6c066e0c8169bb4cfcd1a855903b6e0,2026-10-08 08:05:25,albums,200,49,100,9333,success,null
"{""id"":2,""title"":""sunt qui excepturi placeat culpa"",""userId"":1}",2,ad0a2fc03e55afeaef7307303bbe5894952c45e7a73f29db60e0cc7f83065152,2026-10-08 08:05:25,albums,200,49,100,9333,success,null
"{""id"":3,""title"":""omnis laborum odio"",""userId"":1}",3,641b754a8fb1c4ebcafcbeff03ba4657ccfe89558f473be16ad2f521579c2714,2026-10-08 08:05:25,albums,200,49,100,9333,success,null
"{""id"":4,""title"":""non esse culpa molestiae omnis sed optio"",""userId"":1}",4,1a9b6fc0e57da3a49c848f33ea92bac89c9c9f5ba2d85b8e02e730d1dd54f2aa,2026-10-08 08:05:25,albums,200,49,100,9333,success,null
"{""id"":5,""title"":""eaque aut omnis a"",""userId"":1}",5,be91d30054026e969712dbbdd25291da4a3d55f73064d3f244a0b7b9fa3d7b73,2026-10-08 08:05:25,albums,200,49,100,9333,success,null
"{""id"":6,""title"":""natus impedit quibusdam illo est"",""userId"":1}",6,65f686ccde086c3adc04c8b5694e2f649fba2a6f686635e65cbd391f01c5409f,2026-10-08 08:05:25,albums,200,49,100,9333,success,null
"{""id"":7,""title"":""quibusdam autem aliquid et et quia"",""userId"":1}",7,7b5cea6e228c7edce1ec352a3ad39341fc87453b9b57ae73922bfd9cade5604d,2026-10-08 08:05:25,albums,200,49,100,9333,success,null
"{""id"":8,""title"":""qui fuga est a eum"",""userId"":1}",8,afd0117029a709c9db9c5923276617d96864fd940aceaafcf24dca66518474c7,2026-10-08 08:05:25,albums,200,49,100,9333,success,null
"{""id"":9,""title"":""saepe unde necessitatibus rem"",""userId"":1}",9,d94cb52d1ecfac7009a4521f7dc769c165a1e1871714da1f361fbccf3c19bc74,2026-10-08 08:05:25,albums,200,49,100,9333,success,null
"{""id"":10,""title"":""distinctio laborum qui"",""userId"":1}",10,8db051117cf9b4e1f8cf3ed1f9b99bcfd2179ac688a1b365156b97fdd28f2ada,2026-10-08 08:05:25,albums,200,49,100,9333,success,null


In [0]:
# ── Ingestion Audit Summary ───────────────────────────────────
print("=" * 90)
print("📋 INGESTION AUDIT LOG — DATA QUALITY AT INGESTION")
print("=" * 90)

audit = spark.table(f"{SCHEMAS['bronze']}.ingestion_audit_log")

# Print summary line by line for console readability
audit_rows = audit.collect()
print(f"{'Endpoint':<12} {'Status':<8} {'HTTP':<6} {'Latency':>8} {'Records':>8} {'Size (bytes)':>14}  Error")
print("-" * 90)
for r in audit_rows:
    err = r['error_message'] or ""
    print(f"{r['endpoint']:<12} {r['ingestion_status']:<8} {str(r['http_status_code']):<6} {str(r['response_latency_ms']):>7}ms {str(r['payload_record_count']):>8} {r['payload_size_bytes']:>14,}  {err}")

print("-" * 90)
succeeded = audit.filter(col("ingestion_status") == "success").count()
failed_count = audit.filter(col("ingestion_status") != "success").count()
print(f"✅ {succeeded} endpoints succeeded | ❌ {failed_count} endpoints failed")
print("=" * 90)

# Display the full audit table
audit.display()

📋 INGESTION AUDIT LOG — DATA QUALITY AT INGESTION
Endpoint     Status   HTTP    Latency  Records   Size (bytes)  Error
------------------------------------------------------------------------------------------
users        success  200         72ms       10          5,645  
posts        success  200         57ms      100         27,520  
comments     success  200        114ms      500        157,745  
albums       success  200         47ms      100          9,333  
photos       success  200         78ms     5000      1,039,898  
users        success  200         65ms       10          5,645  
posts        success  200         49ms      100         27,520  
comments     success  200         48ms      500        157,745  
albums       success  200         51ms      100          9,333  
photos       success  200         50ms     5000      1,039,898  
users        success  200         68ms       10          5,645  
posts        success  200         43ms      100         27,520  
comments  

endpoint,ingested_at,http_status_code,response_latency_ms,payload_record_count,payload_size_bytes,ingestion_status,error_message
users,2026-10-14 09:18:25,200,72,10,5645,success,null
posts,2026-10-14 09:18:25,200,57,100,27520,success,null
comments,2026-10-14 09:18:25,200,114,500,157745,success,null
albums,2026-10-14 09:18:25,200,47,100,9333,success,null
photos,2026-10-14 09:18:25,200,78,5000,1039898,success,null
users,2026-10-06 06:12:42,200,65,10,5645,success,null
posts,2026-10-06 06:12:42,200,49,100,27520,success,null
comments,2026-10-06 06:12:42,200,48,500,157745,success,null
albums,2026-10-06 06:12:42,200,51,100,9333,success,null
photos,2026-10-06 06:12:42,200,50,5000,1039898,success,null


## ⚪ Silver Layer — Cleansed & Conformed

Parse the raw JSON payloads into typed columns, flatten nested structures, remove duplicates, and add audit columns (`processed_at`, `source_endpoint`).

In [0]:
PROCESSED_TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

/home/spark-c23eba55-7041-4e7a-a511-fc/.ipykernel/66/command-5596105659962686-3559678691:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  PROCESSED_TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {SCHEMAS['silver']}.users")

In [0]:


# ── Silver: Users ─────────────────────────────────────────────
raw_users = spark.table(f"{SCHEMAS['bronze']}.raw_users")

silver_users = (
    raw_users
    .select(
        from_json(col("raw_payload"),
            StructType([
                StructField("id", IntegerType()),
                StructField("name", StringType()),
                StructField("username", StringType()),
                StructField("email", StringType()),
                StructField("phone", StringType()),
                StructField("website", StringType()),
                StructField("company", StructType([
                    StructField("name", StringType()),
                    StructField("catchPhrase", StringType()),
                    StructField("bs", StringType()),
                ])),
                StructField("address", StructType([
                    StructField("street", StringType()),
                    StructField("suite", StringType()),
                    StructField("city", StringType()),
                    StructField("zipcode", StringType()),
                    StructField("geo", StructType([
                        StructField("lat", StringType()),
                        StructField("lng", StringType()),
                    ])),
                ])),
            ])
        ).alias("data")
    )
    .select(
        col("data.id").alias("user_id"),
        col("data.name").alias("name"),
        col("data.username").alias("username"),
        col("data.email").alias("email"),
        col("data.phone").alias("phone"),
        col("data.website").alias("website"),
        col("data.company.name").alias("company_name"),
        col("data.company.catchPhrase").alias("company_catchphrase"),
        col("data.address.street").alias("address_street"),
        col("data.address.city").alias("address_city"),
        col("data.address.zipcode").alias("address_zipcode"),
        col("data.address.geo.lat").alias("geo_lat"),
        col("data.address.geo.lng").alias("geo_lng"),
    )
    .dropDuplicates(["user_id"])
    .withColumn("processed_at", lit(PROCESSED_TS))
    .withColumn("source_endpoint", lit("users"))
)

silver_users = silver_users.withColumn(
    "data_quality_flag",
    when(col("geo_lat").isNull()| col("geo_lng").isNull(),"INVALID_GEO_LOCATION")
    .when(col("email").isNull()| (col("email") == "") |  (~col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")),"INVALID_EMAIL" )
    .otherwise("VALID")
)

silver_users = silver_users.withColumn(
    "email_domain",concat(lit("@"), split(col("email"),"@")[1])
)

(
    silver_users.write.format("delta").mode("overwrite")
    .saveAsTable(f"{SCHEMAS['silver']}.users")
)
print(f"✅ Silver users table: {silver_users.count()} rows")
silver_users.display()

✅ Silver users table: 10 rows


user_id,name,username,email,phone,website,company_name,company_catchphrase,address_street,address_city,address_zipcode,geo_lat,geo_lng,processed_at,source_endpoint,data_quality_flag,email_domain
1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Romaguera-Crona,Multi-layered client-server neural-net,Kulas Light,Gwenborough,92998-3874,-37.3159,81.1496,2026-10-14 10:48:42,users,VALID,@april.biz
2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Deckow-Crist,Proactive didactic contingency,Victor Plains,Wisokyburgh,90566-7771,-43.9509,-34.4618,2026-10-14 10:48:42,users,VALID,@melissa.tv
3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Romaguera-Jacobson,Face to face bifurcated interface,Douglas Extension,McKenziehaven,59590-4157,-68.6102,-47.0653,2026-10-14 10:48:42,users,VALID,@yesenia.net
4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Robel-Corkery,Multi-tiered zero tolerance productivity,Hoeger Mall,South Elvis,53919-4257,29.4572,-164.2990,2026-10-14 10:48:42,users,VALID,@kory.org
5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Keebler LLC,User-centric fault-tolerant solution,Skiles Walks,Roscoeview,33263,-31.8129,62.5342,2026-10-14 10:48:42,users,VALID,@annie.ca
6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,1-477-935-8478 x6430,ola.org,Considine-Lockman,Synchronised bottom-line interface,Norberto Crossing,South Christy,23505-1337,-71.4197,71.7478,2026-10-14 10:48:42,users,VALID,@jasper.info
7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,210.067.6132,elvis.io,Johns Group,Configurable multimedia task-force,Rex Trail,Howemouth,58804-1099,24.8918,21.8984,2026-10-14 10:48:42,users,VALID,@billy.biz
8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,586.493.6943 x140,jacynthe.com,Abernathy Group,Implemented secondary concept,Ellsworth Summit,Aliyaview,45169,-14.3990,-120.7677,2026-10-14 10:48:42,users,VALID,@rosamond.me
9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,(775)976-6794 x41206,conrad.com,Yost and Sons,Switchable contextually-based project,Dayna Park,Bartholomebury,76495-3109,24.6463,-168.8889,2026-10-14 10:48:42,users,VALID,@dana.io
10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,024-648-3804,ambrose.net,Hoeger LLC,Centralized empowering task-force,Kattie Turnpike,Lebsackbury,31428-2261,-38.2386,57.2232,2026-10-14 10:48:42,users,VALID,@karina.biz


In [0]:
# ── Silver: Posts ─────────────────────────────────────────────
raw_posts = spark.table(f"{SCHEMAS['bronze']}.raw_posts")

silver_posts = (
    raw_posts
    .select(
        from_json(col("raw_payload"),
            StructType([
                StructField("userId", IntegerType()),
                StructField("id", IntegerType()),
                StructField("title", StringType()),
                StructField("body", StringType()),
            ])
        ).alias("data")
    )
    .select(
        col("data.id").alias("post_id"),
        col("data.userId").alias("user_id"),
        col("data.title").alias("title"),
        col("data.body").alias("body"),
    )
    .dropDuplicates(["post_id"])
    .withColumn("processed_at", lit(PROCESSED_TS))
    .withColumn("source_endpoint", lit("posts"))
)

(
    silver_posts.write.format("delta").mode("overwrite")
    .saveAsTable(f"{SCHEMAS['silver']}.posts")
)
print(f"✅ Silver posts table: {silver_posts.count()} rows")
silver_posts.display()

✅ Silver posts table: 100 rows


post_id,user_id,title,body,processed_at,source_endpoint
1,1,sunt aut facere repellat provident occaecati excepturi optio reprehenderit,quia et suscipit suscipit recusandae consequuntur expedita et cum reprehenderit molestiae ut ut quas totam nostrum rerum est autem sunt rem eveniet architecto,2026-10-14 10:48:42,posts
2,1,qui est esse,est rerum tempore vitae sequi sint nihil reprehenderit dolor beatae ea dolores neque fugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis qui aperiam non debitis possimus qui neque nisi nulla,2026-10-14 10:48:42,posts
3,1,ea molestias quasi exercitationem repellat qui ipsa sit aut,et iusto sed quo iure voluptatem occaecati omnis eligendi aut ad voluptatem doloribus vel accusantium quis pariatur molestiae porro eius odio et labore et velit aut,2026-10-14 10:48:42,posts
4,1,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci sit amet autem assumenda provident rerum culpa quis hic commodi nesciunt rem tenetur doloremque ipsam iure quis sunt voluptatem rerum illo velit,2026-10-14 10:48:42,posts
5,1,nesciunt quas odio,repudiandae veniam quaerat sunt sed alias aut fugiat sit autem sed est voluptatem omnis possimus esse voluptatibus quis est aut tenetur dolor neque,2026-10-14 10:48:42,posts
6,1,dolorem eum magni eos aperiam quia,ut aspernatur corporis harum nihil quis provident sequi mollitia nobis aliquid molestiae perspiciatis et ea nemo ab reprehenderit accusantium quas voluptate dolores velit et doloremque molestiae,2026-10-14 10:48:42,posts
7,1,magnam facilis autem,dolore placeat quibusdam ea quo vitae magni quis enim qui quis quo nemo aut saepe quidem repellat excepturi ut quia sunt ut sequi eos ea sed quas,2026-10-14 10:48:42,posts
8,1,dolorem dolore est ipsam,dignissimos aperiam dolorem qui eum facilis quibusdam animi sint suscipit qui sint possimus cum quaerat magni maiores excepturi ipsam ut commodi dolor voluptatum modi aut vitae,2026-10-14 10:48:42,posts
9,1,nesciunt iure omnis dolorem tempora et accusantium,consectetur animi nesciunt iure dolore enim quia ad veniam autem ut quam aut nobis et est aut quod aut provident voluptas autem voluptas,2026-10-14 10:48:42,posts
10,1,optio molestias id quia eum,quo et expedita modi cum officia vel magni doloribus qui repudiandae vero nisi sit quos veniam quod sed accusamus veritatis error,2026-10-14 10:48:42,posts


In [0]:
# ── Silver: Comments ──────────────────────────────────────────
raw_comments = spark.table(f"{SCHEMAS['bronze']}.raw_comments")

silver_comments = (
    raw_comments
    .select(
        from_json(col("raw_payload"),
            StructType([
                StructField("postId", IntegerType()),
                StructField("id", IntegerType()),
                StructField("name", StringType()),
                StructField("email", StringType()),
                StructField("body", StringType()),
            ])
        ).alias("data")
    )
    .select(
        col("data.id").alias("comment_id"),
        col("data.postId").alias("post_id"),
        col("data.name").alias("comment_name"),
        col("data.email").alias("commenter_email"),
        col("data.body").alias("comment_body"),
    )
    .dropDuplicates(["comment_id"])
    .withColumn("processed_at", lit(PROCESSED_TS))
    .withColumn("source_endpoint", lit("comments"))
)

(
    silver_comments.write.format("delta").mode("overwrite")
    .saveAsTable(f"{SCHEMAS['silver']}.comments")
)
print(f"✅ Silver comments table: {silver_comments.count()} rows")
silver_comments.display()

✅ Silver comments table: 500 rows


comment_id,post_id,comment_name,commenter_email,comment_body,processed_at,source_endpoint
1,1,id labore ex et quam laborum,Eliseo@gardner.biz,laudantium enim quasi est quidem magnam voluptate ipsam eos tempora quo necessitatibus dolor quam autem quasi reiciendis et nam sapiente accusantium,2026-10-14 10:48:42,comments
2,1,quo vero reiciendis velit similique earum,Jayne_Kuhic@sydney.com,est natus enim nihil est dolore omnis voluptatem numquam et omnis occaecati quod ullam at voluptatem error expedita pariatur nihil sint nostrum voluptatem reiciendis et,2026-10-14 10:48:42,comments
3,1,odio adipisci rerum aut animi,Nikita@garfield.biz,quia molestiae reprehenderit quasi aspernatur aut expedita occaecati aliquam eveniet laudantium omnis quibusdam delectus saepe quia accusamus maiores nam est cum et ducimus et vero voluptates excepturi deleniti ratione,2026-10-14 10:48:42,comments
4,1,alias odio sit,Lew@alysha.tv,non et atque occaecati deserunt quas accusantium unde odit nobis qui voluptatem quia voluptas consequuntur itaque dolor et qui rerum deleniti ut occaecati,2026-10-14 10:48:42,comments
5,1,vero eaque aliquid doloribus et culpa,Hayden@althea.biz,harum non quasi et ratione tempore iure ex voluptates in ratione harum architecto fugit inventore cupiditate voluptates magni quo et,2026-10-14 10:48:42,comments
6,2,et fugit eligendi deleniti quidem qui sint nihil autem,Presley.Mueller@myrl.com,doloribus at sed quis culpa deserunt consectetur qui praesentium accusamus fugiat dicta voluptatem rerum ut voluptate autem voluptatem repellendus aspernatur dolorem in,2026-10-14 10:48:42,comments
7,2,repellat consequatur praesentium vel minus molestias voluptatum,Dallas@ole.me,maiores sed dolores similique labore et inventore et quasi temporibus esse sunt id et eos voluptatem aliquam aliquid ratione corporis molestiae mollitia quia et magnam dolor,2026-10-14 10:48:42,comments
8,2,et omnis dolorem,Mallory_Kunze@marie.org,ut voluptatem corrupti velit ad voluptatem maiores et nisi velit vero accusamus maiores voluptates quia aliquid ullam eaque,2026-10-14 10:48:42,comments
9,2,provident id voluptas,Meghan_Littel@rene.us,sapiente assumenda molestiae atque adipisci laborum distinctio aperiam et ab ut omnis et occaecati aspernatur odit sit rem expedita quas enim ipsam minus,2026-10-14 10:48:42,comments
10,2,eaque et deleniti atque tenetur ut quo ut,Carmen_Keeling@caroline.name,voluptate iusto quis nobis reprehenderit ipsum amet nulla quia quas dolores velit et non aut quia necessitatibus nostrum quaerat nulla et accusamus nisi facilis,2026-10-14 10:48:42,comments


In [0]:
raw_albums =spark.table(f"{SCHEMAS['bronze']}.raw_albums")
# raw_albums.display()    
silver_albums = (
    raw_albums
    .select(
        from_json(
            col("raw_payload"),
            StructType([
                StructField("id", StringType()),
                StructField("userId", StringType()),
                StructField("title", StringType())
            ])
        ).alias("data")
    )
    .select(
        col("data.id").alias("album_id"),
        col("data.userId").alias("user_id"),
        col("data.title").alias("title")
    ).dropDuplicates(["album_id"])
    .withColumn("processed_at",lit(PROCESSED_TS))
    .withColumn("source_endpoint",lit("albums"))
)
# silver_albums.display()

(
    silver_albums.write.format("delta").mode("overwrite")
    .saveAsTable(f"{SCHEMAS['silver']}.albums")
)

print(f"✅ Silver comments table:{silver_albums.count()} rows")
silver_albums.display()

✅ Silver comments table:100 rows


album_id,user_id,title,processed_at,source_endpoint
1,1,quidem molestiae enim,2026-10-14 10:48:42,albums
2,1,sunt qui excepturi placeat culpa,2026-10-14 10:48:42,albums
3,1,omnis laborum odio,2026-10-14 10:48:42,albums
4,1,non esse culpa molestiae omnis sed optio,2026-10-14 10:48:42,albums
5,1,eaque aut omnis a,2026-10-14 10:48:42,albums
6,1,natus impedit quibusdam illo est,2026-10-14 10:48:42,albums
7,1,quibusdam autem aliquid et et quia,2026-10-14 10:48:42,albums
8,1,qui fuga est a eum,2026-10-14 10:48:42,albums
9,1,saepe unde necessitatibus rem,2026-10-14 10:48:42,albums
10,1,distinctio laborum qui,2026-10-14 10:48:42,albums


In [0]:
raw_photo = spark.table(f"{SCHEMAS['bronze']}.raw_photos")
# raw_photo.display()
silver_photo = (
    raw_photo.select(
        from_json(
            col("raw_payload"),
            StructType([
                StructField("id", StringType()),
                StructField("albumId", StringType()),
                StructField("title", StringType()),
                StructField("url", StringType()),
                StructField("thumbnailUrl", StringType())
            ])
        ).alias("data")
    )
    .select(
        col("data.id").alias("photo_id"),
        col("data.albumId").alias("album_id"),
        col("data.title").alias("title"),
        col("data.url").alias("url"),
        col("data.thumbnailUrl").alias("thumbnailUrl")
    )
    .dropDuplicates(["photo_id"])
    .withColumn("processed_at", lit(PROCESSED_TS))
    .withColumn("source_endpoint", lit("raw_photos"))
)

(
    silver_photo.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMAS['silver']}.photo")
)

print(f"✅ Silver comments table: {silver_photo.count()}rows")
silver_photo.display()

✅ Silver comments table: 5000rows


photo_id,album_id,title,url,thumbnailUrl,processed_at,source_endpoint
4012,81,quo dolorem repellat,https://picsum.photos/seed/4012/600,https://picsum.photos/seed/4012/150,2026-10-14 10:48:42,raw_photos
4025,81,rerum quidem omnis facilis voluptate est deserunt,https://picsum.photos/seed/4025/600,https://picsum.photos/seed/4025/150,2026-10-14 10:48:42,raw_photos
4032,81,dolor ut animi ipsam occaecati aut,https://picsum.photos/seed/4032/600,https://picsum.photos/seed/4032/150,2026-10-14 10:48:42,raw_photos
4053,82,id quis nulla suscipit assumenda commodi natus illo est,https://picsum.photos/seed/4053/600,https://picsum.photos/seed/4053/150,2026-10-14 10:48:42,raw_photos
4074,82,aut impedit eaque similique non et et,https://picsum.photos/seed/4074/600,https://picsum.photos/seed/4074/150,2026-10-14 10:48:42,raw_photos
4076,82,dolor culpa possimus suscipit mollitia consectetur eveniet totam omnis,https://picsum.photos/seed/4076/600,https://picsum.photos/seed/4076/150,2026-10-14 10:48:42,raw_photos
4090,82,atque et deleniti fugit asperiores vel quia vitae,https://picsum.photos/seed/4090/600,https://picsum.photos/seed/4090/150,2026-10-14 10:48:42,raw_photos
4100,82,porro quod quos,https://picsum.photos/seed/4100/600,https://picsum.photos/seed/4100/150,2026-10-14 10:48:42,raw_photos
4120,83,esse atque et impedit,https://picsum.photos/seed/4120/600,https://picsum.photos/seed/4120/150,2026-10-14 10:48:42,raw_photos
4121,83,aut qui repellat,https://picsum.photos/seed/4121/600,https://picsum.photos/seed/4121/150,2026-10-14 10:48:42,raw_photos


## 🟡 Gold Layer — Analytics-Ready

Business-level aggregations built from the silver tables:

1. **posts_per_user** — number of posts and total comments per user
2. **comments_per_post** — comment count and unique commenters per post
3. **user_activity_summary** — comprehensive user engagement dashboard

In [0]:
# ── Gold: Posts per User ──────────────────────────────────────
u = spark.table(f"{SCHEMAS['silver']}.users")
p = spark.table(f"{SCHEMAS['silver']}.posts")
c = spark.table(f"{SCHEMAS['silver']}.comments")

posts_per_user = (
    u.join(p, "user_id", "left")
     .join(c, "post_id", "left")
     .groupBy(u.user_id, u.name, u.username, u.company_name, u.address_city)
     .agg(
         countDistinct(p.post_id).alias("total_posts"),
         countDistinct(c.comment_id).alias("total_comments_received"),
         countDistinct(c.commenter_email).alias("unique_commenters"),
     )
     .orderBy(col("total_posts").desc(), col("total_comments_received").desc())
)

(
    posts_per_user.write.format("delta").mode("overwrite")
    .saveAsTable(f"{SCHEMAS['gold']}.posts_per_user")
)
print(f"✅ Gold posts_per_user: {posts_per_user.count()} rows")
posts_per_user.display()

✅ Gold posts_per_user: 10 rows


user_id,name,username,company_name,address_city,total_posts,total_comments_received,unique_commenters
3,Clementine Bauch,Samantha,Romaguera-Jacobson,McKenziehaven,10,50,50
4,Patricia Lebsack,Karianne,Robel-Corkery,South Elvis,10,50,50
2,Ervin Howell,Antonette,Deckow-Crist,Wisokyburgh,10,50,50
1,Leanne Graham,Bret,Romaguera-Crona,Gwenborough,10,50,50
6,Mrs. Dennis Schulist,Leopoldo_Corkery,Considine-Lockman,South Christy,10,50,50
10,Clementina DuBuque,Moriah.Stanton,Hoeger LLC,Lebsackbury,10,50,50
8,Nicholas Runolfsdottir V,Maxime_Nienow,Abernathy Group,Aliyaview,10,50,50
7,Kurtis Weissnat,Elwyn.Skiles,Johns Group,Howemouth,10,50,50
9,Glenna Reichert,Delphine,Yost and Sons,Bartholomebury,10,50,50
5,Chelsey Dietrich,Kamren,Keebler LLC,Roscoeview,10,50,50


In [0]:
u.display()

user_id,name,username,email,phone,website,company_name,company_catchphrase,address_street,address_city,address_zipcode,geo_lat,geo_lng,processed_at,source_endpoint,data_quality_flag,email_domain
5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Keebler LLC,User-centric fault-tolerant solution,Skiles Walks,Roscoeview,33263,-31.8129,62.5342,2026-10-14 10:48:42,users,VALID,@annie.ca
10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,024-648-3804,ambrose.net,Hoeger LLC,Centralized empowering task-force,Kattie Turnpike,Lebsackbury,31428-2261,-38.2386,57.2232,2026-10-14 10:48:42,users,VALID,@karina.biz
1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Romaguera-Crona,Multi-layered client-server neural-net,Kulas Light,Gwenborough,92998-3874,-37.3159,81.1496,2026-10-14 10:48:42,users,VALID,@april.biz
3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Romaguera-Jacobson,Face to face bifurcated interface,Douglas Extension,McKenziehaven,59590-4157,-68.6102,-47.0653,2026-10-14 10:48:42,users,VALID,@yesenia.net
2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Deckow-Crist,Proactive didactic contingency,Victor Plains,Wisokyburgh,90566-7771,-43.9509,-34.4618,2026-10-14 10:48:42,users,VALID,@melissa.tv
6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,1-477-935-8478 x6430,ola.org,Considine-Lockman,Synchronised bottom-line interface,Norberto Crossing,South Christy,23505-1337,-71.4197,71.7478,2026-10-14 10:48:42,users,VALID,@jasper.info
9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,(775)976-6794 x41206,conrad.com,Yost and Sons,Switchable contextually-based project,Dayna Park,Bartholomebury,76495-3109,24.6463,-168.8889,2026-10-14 10:48:42,users,VALID,@dana.io
7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,210.067.6132,elvis.io,Johns Group,Configurable multimedia task-force,Rex Trail,Howemouth,58804-1099,24.8918,21.8984,2026-10-14 10:48:42,users,VALID,@billy.biz
4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Robel-Corkery,Multi-tiered zero tolerance productivity,Hoeger Mall,South Elvis,53919-4257,29.4572,-164.2990,2026-10-14 10:48:42,users,VALID,@kory.org
8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,586.493.6943 x140,jacynthe.com,Abernathy Group,Implemented secondary concept,Ellsworth Summit,Aliyaview,45169,-14.3990,-120.7677,2026-10-14 10:48:42,users,VALID,@rosamond.me


In [0]:
a= spark.table(f"{SCHEMAS['silver']}.albums")
a.display()

album_id,user_id,title,processed_at,source_endpoint
56,6,omnis qui maiores tempora officiis omnis rerum sed repellat,2026-10-14 10:48:42,albums
73,8,quos omnis officia,2026-10-14 10:48:42,albums
14,2,ducimus molestias eos animi atque nihil,2026-10-14 10:48:42,albums
16,2,voluptatem aut maxime inventore autem magnam atque repellat,2026-10-14 10:48:42,albums
35,4,et impedit nisi quae magni necessitatibus sed aut pariatur,2026-10-14 10:48:42,albums
52,6,necessitatibus quas et sunt at voluptatem,2026-10-14 10:48:42,albums
55,6,qui culpa itaque omnis in nesciunt architecto error,2026-10-14 10:48:42,albums
62,7,eos ea non recusandae iste ut quasi,2026-10-14 10:48:42,albums
64,7,autem voluptatem amet iure quae,2026-10-14 10:48:42,albums
89,9,sit optio id voluptatem est eum et,2026-10-14 10:48:42,albums


In [0]:
a= spark.table(f"{SCHEMAS['silver']}.albums")


albums_per_user =(
    u.join(a,"user_id","left")
    .groupBy(u.user_id,u.username,u.name,u.company_name)
    .agg(
        countDistinct(a.album_id).alias("total_albums")
    ).orderBy(col("total_albums").desc())
)

(
    albums_per_user.write.format("delta").mode("overwrite")
    .saveAsTable(f"{SCHEMAS['gold']}.albums_per_user")
)

print(f"✅ Gold albums_per_user: {albums_per_user.count()} rows")
albums_per_user.display()

✅ Gold albums_per_user: 10 rows


user_id,username,name,company_name,total_albums
3,Samantha,Clementine Bauch,Romaguera-Jacobson,10
1,Bret,Leanne Graham,Romaguera-Crona,10
2,Antonette,Ervin Howell,Deckow-Crist,10
10,Moriah.Stanton,Clementina DuBuque,Hoeger LLC,10
5,Kamren,Chelsey Dietrich,Keebler LLC,10
7,Elwyn.Skiles,Kurtis Weissnat,Johns Group,10
9,Delphine,Glenna Reichert,Yost and Sons,10
8,Maxime_Nienow,Nicholas Runolfsdottir V,Abernathy Group,10
6,Leopoldo_Corkery,Mrs. Dennis Schulist,Considine-Lockman,10
4,Karianne,Patricia Lebsack,Robel-Corkery,10


In [0]:
ph =spark.table(f"{SCHEMAS['silver']}.photo")

photos_per_albums =(
    a.join(ph,"album_id","left")
    .join(u,"user_id","left")
    .groupBy(
        a.album_id,
        a.title,
        u.name,
        a.user_id
    )
    .agg(
        countDistinct(ph.photo_id).alias("total_photos"),
        avg(length(ph.title)).alias("average_title_length")
    )

)

window_spec = (
    Window.partitionBy('user_id').orderBy(
        col("total_photos").desc(),col("album_id").asc()
    )
)

photos_per_albums =(
    photos_per_albums.withColumn("rank",row_number().over(window_spec)).filter(col("rank") == 1)
)

(
    photos_per_albums.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMAS['gold']}.photos_per_albums")
)


In [0]:
p.display()

post_id,user_id,title,body,processed_at,source_endpoint
12,2,in quibusdam tempore odit est dolorem,itaque id aut magnam praesentium quia et ea odit et ea voluptas et sapiente quia nihil amet occaecati quia id voluptatem incidunt ea est distinctio odio,2026-10-14 10:48:42,posts
18,2,voluptate et itaque vero tempora molestiae,eveniet quo quis laborum totam consequatur non dolor ut et est repudiandae est voluptatem vel debitis et magnam,2026-10-14 10:48:42,posts
38,4,explicabo et eos deleniti nostrum ab id repellendus,animi esse sit aut sit nesciunt assumenda eum voluptas quia voluptatibus provident quia necessitatibus ea rerum repudiandae quia voluptatem delectus fugit aut id quia ratione optio eos iusto veniam iure,2026-10-14 10:48:42,posts
67,7,aliquid eos sed fuga est maxime repellendus,reprehenderit id nostrum voluptas doloremque pariatur sint et accusantium quia quod aspernatur et fugiat amet non sapiente et consequatur necessitatibus molestiae,2026-10-14 10:48:42,posts
70,7,voluptatem laborum magni,sunt repellendus quae est asperiores aut deleniti esse accusamus repellendus quia aut quia dolorem unde eum tempora esse dolore,2026-10-14 10:48:42,posts
93,10,beatae soluta recusandae,dolorem quibusdam ducimus consequuntur dicta aut quo laboriosam voluptatem quis enim recusandae ut sed sunt nostrum est odit totam sit error sed sunt eveniet provident qui nulla,2026-10-14 10:48:42,posts
16,2,sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio,suscipit nam nisi quo aperiam aut asperiores eos fugit maiores voluptatibus quia voluptatem quis ullam qui in alias quia est consequatur magni mollitia accusamus ea nisi voluptate dicta,2026-10-14 10:48:42,posts
64,7,et fugit quas eum in in aperiam quod,id velit blanditiis eum ea voluptatem molestiae sint occaecati est eos perspiciatis incidunt a error provident eaque aut aut qui,2026-10-14 10:48:42,posts
74,8,enim unde ratione doloribus quas enim ut sit sapiente,odit qui et et necessitatibus sint veniam mollitia amet doloremque molestiae commodi similique magnam et quam blanditiis est itaque quo et tenetur ratione occaecati molestiae tempora,2026-10-14 10:48:42,posts
94,10,qui qui voluptates illo iste minima,aspernatur expedita soluta quo ab ut similique expedita dolores amet sed temporibus distinctio magnam saepe deleniti omnis facilis nam ipsum natus sint similique omnis,2026-10-14 10:48:42,posts


In [0]:
u.display()

user_id,name,username,email,phone,website,company_name,company_catchphrase,address_street,address_city,address_zipcode,geo_lat,geo_lng,processed_at,source_endpoint,data_quality_flag,email_domain
5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Keebler LLC,User-centric fault-tolerant solution,Skiles Walks,Roscoeview,33263,-31.8129,62.5342,2026-10-14 10:48:42,users,VALID,@annie.ca
10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,024-648-3804,ambrose.net,Hoeger LLC,Centralized empowering task-force,Kattie Turnpike,Lebsackbury,31428-2261,-38.2386,57.2232,2026-10-14 10:48:42,users,VALID,@karina.biz
1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Romaguera-Crona,Multi-layered client-server neural-net,Kulas Light,Gwenborough,92998-3874,-37.3159,81.1496,2026-10-14 10:48:42,users,VALID,@april.biz
3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Romaguera-Jacobson,Face to face bifurcated interface,Douglas Extension,McKenziehaven,59590-4157,-68.6102,-47.0653,2026-10-14 10:48:42,users,VALID,@yesenia.net
2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Deckow-Crist,Proactive didactic contingency,Victor Plains,Wisokyburgh,90566-7771,-43.9509,-34.4618,2026-10-14 10:48:42,users,VALID,@melissa.tv
6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,1-477-935-8478 x6430,ola.org,Considine-Lockman,Synchronised bottom-line interface,Norberto Crossing,South Christy,23505-1337,-71.4197,71.7478,2026-10-14 10:48:42,users,VALID,@jasper.info
9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,(775)976-6794 x41206,conrad.com,Yost and Sons,Switchable contextually-based project,Dayna Park,Bartholomebury,76495-3109,24.6463,-168.8889,2026-10-14 10:48:42,users,VALID,@dana.io
7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,210.067.6132,elvis.io,Johns Group,Configurable multimedia task-force,Rex Trail,Howemouth,58804-1099,24.8918,21.8984,2026-10-14 10:48:42,users,VALID,@billy.biz
4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Robel-Corkery,Multi-tiered zero tolerance productivity,Hoeger Mall,South Elvis,53919-4257,29.4572,-164.2990,2026-10-14 10:48:42,users,VALID,@kory.org
8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,586.493.6943 x140,jacynthe.com,Abernathy Group,Implemented secondary concept,Ellsworth Summit,Aliyaview,45169,-14.3990,-120.7677,2026-10-14 10:48:42,users,VALID,@rosamond.me


In [0]:
display(a.limit(5))

album_id,user_id,title,processed_at,source_endpoint
56,6,omnis qui maiores tempora officiis omnis rerum sed repellat,2026-10-14 10:48:42,albums
73,8,quos omnis officia,2026-10-14 10:48:42,albums
14,2,ducimus molestias eos animi atque nihil,2026-10-14 10:48:42,albums
16,2,voluptatem aut maxime inventore autem magnam atque repellat,2026-10-14 10:48:42,albums
35,4,et impedit nisi quae magni necessitatibus sed aut pariatur,2026-10-14 10:48:42,albums


In [0]:
 # User engagement composite score — Combine posts, comments received, albums, and photos into a single weighted engagement score per user. Rank users by this score to identify the most "active" mock users.

 

In [0]:
# ── Gold: Comments per Post ───────────────────────────────────
comments_per_post = (
    p.join(c, "post_id", "left")
     .join(u, "user_id", "left")
     .groupBy(p.post_id, p.title, u.name.alias("author_name"))
     .agg(
         countDistinct(c.comment_id).alias("total_comments"),
         countDistinct(c.commenter_email).alias("unique_commenters"),
         collect_set(c.commenter_email).alias("commenter_emails"),
     )
     .orderBy(col("total_comments").desc())
)

(
    comments_per_post.write.format("delta").mode("overwrite")
    .saveAsTable(f"{SCHEMAS['gold']}.comments_per_post")
)
print(f"✅ Gold comments_per_post: {comments_per_post.count()} rows")
comments_per_post.display()

✅ Gold comments_per_post: 100 rows


post_id,title,author_name,total_comments,unique_commenters,commenter_emails
12,in quibusdam tempore odit est dolorem,Ervin Howell,5,5,"List(Americo@estrella.net, Vince_Crist@heidi.biz, Darron.Nikolaus@eulah.me, Jameson@tony.info, Ezra_Abshire@lyda.us)"
70,voluptatem laborum magni,Kurtis Weissnat,5,5,"List(Loren@aric.biz, Arjun@natalie.ca, Kirstin@tina.info, Bradley.Lang@marilyne.tv, Anthony.Koepp@savannah.tv)"
1,sunt aut facere repellat provident occaecati excepturi optio reprehenderit,Leanne Graham,5,5,"List(Lew@alysha.tv, Hayden@althea.biz, Jayne_Kuhic@sydney.com, Nikita@garfield.biz, Eliseo@gardner.biz)"
32,doloremque illum aliquid sunt,Patricia Lebsack,5,5,"List(Mitchel.Williamson@fletcher.io, Makenzie@libbie.io, Telly_Lynch@karl.co.uk, Amiya@perry.us, Meghan@akeem.tv)"
51,soluta aliquam aperiam consequatur illo quis voluptas,Mrs. Dennis Schulist,5,5,"List(Jayde@geovanny.io, Delta_Welch@carleton.tv, Ardella@khalid.biz, Carissa.Krajcik@jean.name, Ayden_Hickle@stephany.tv)"
55,sit vel voluptatem et non libero,Mrs. Dennis Schulist,5,5,"List(Micaela.Powlowski@saul.me, Ethelyn.Moore@gabe.info, Imelda_Klein@melany.biz, Orland@larry.name, Evangeline_Kuvalis@santina.ca)"
94,qui qui voluptates illo iste minima,Clementina DuBuque,5,5,"List(Adrianna_Howell@molly.io, Amiya.Morar@emma.tv, Katarina.Wolff@joel.io, Destany@bailey.info, Brant@yasmin.co.uk)"
27,quasi id et eos tenetur aut quo autem,Clementine Bauch,5,5,"List(Emmet@guy.biz, Megane.Fritsch@claude.name, Jasen_Rempel@willis.org, Harmony@reggie.com, Amya@durward.ca)"
34,magnam ut rerum iure,Patricia Lebsack,5,5,"List(Lee@dawn.net, Gabrielle@jada.co.uk, Eloy@vladimir.com, Aimee.Bins@braeden.ca, Gideon.Hyatt@jalen.tv)"
69,fugiat quod pariatur odit minima,Kurtis Weissnat,5,5,"List(Zander@santino.net, April@larissa.co.uk, Cordelia.Oberbrunner@peyton.com, Glenna_Waters@retha.me, Camila_Runolfsdottir@tressa.tv)"


In [0]:
# ── Gold: User Activity Summary ──────────────────────────────
# A comprehensive engagement summary combining user profile with their posting activity
user_activity = (
    u.join(p, "user_id", "left")
     .join(c, p.post_id == c.post_id, "left")
     .groupBy(
         u.user_id, u.name, u.username, u.email, u.company_name,
         u.address_city, u.website,
     )
     .agg(
         countDistinct(p.post_id).alias("total_posts"),
         countDistinct(c.comment_id).alias("total_comments_on_posts"),
         countDistinct(c.commenter_email).alias("unique_engaged_users"),
         avg(length(p.body)).alias("avg_post_body_length"),
         max(p.title).alias("latest_post_title"),
     )
     .orderBy(col("total_comments_on_posts").desc())
)

(
    user_activity.write.format("delta").mode("overwrite")
    .saveAsTable(f"{SCHEMAS['gold']}.user_activity_summary")
)
print(f"✅ Gold user_activity_summary: {user_activity.count()} rows")
user_activity.display()

✅ Gold user_activity_summary: 10 rows


user_id,name,username,email,company_name,address_city,website,total_posts,total_comments_on_posts,unique_engaged_users,avg_post_body_length,latest_post_title
9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,Yost and Sons,Bartholomebury,conrad.com,10,50,50,154.6,tempora rem veritatis voluptas quo dolores vero
7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,Johns Group,Howemouth,elvis.io,10,50,50,163.5,voluptatem laborum magni
8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,Abernathy Group,Aliyaview,jacynthe.com,10,50,50,160.6,sint hic doloribus consequatur eos non id
6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,Considine-Lockman,South Christy,ola.org,10,50,50,147.0,voluptatum itaque dolores nisi et quasi
1,Leanne Graham,Bret,Sincere@april.biz,Romaguera-Crona,Gwenborough,hildegard.org,10,50,50,164.5,sunt aut facere repellat provident occaecati excepturi optio reprehenderit
5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,Keebler LLC,Roscoeview,demarco.info,10,50,50,162.5,ut voluptatem illum ea doloribus itaque eos
3,Clementine Bauch,Samantha,Nathan@yesenia.net,Romaguera-Jacobson,McKenziehaven,ramiro.info,10,50,50,153.8,rem alias distinctio quo quis
4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,Robel-Corkery,South Elvis,kale.biz,10,50,50,182.3,ullam ut quidem id aut vel consequuntur
10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,Hoeger LLC,Lebsackbury,ambrose.net,10,50,50,154.7,temporibus sit alias delectus eligendi possimus magni
2,Ervin Howell,Antonette,Shanna@melissa.tv,Deckow-Crist,Wisokyburgh,anastasia.net,10,50,50,162.9,voluptatem eligendi optio


In [0]:
# ── Verify All Tables ─────────────────────────────────────────
print("=" * 60)
print("📋 MEDALLION ARCHITECTURE — TABLE INVENTORY")
print("=" * 60)

all_tables = [
    ("BRONZE", f"{SCHEMAS['bronze']}.raw_users"),
    ("BRONZE", f"{SCHEMAS['bronze']}.raw_posts"),
    ("BRONZE", f"{SCHEMAS['bronze']}.raw_comments"),
    ("BRONZE", f"{SCHEMAS['bronze']}.ingestion_audit_log"),
    ("SILVER", f"{SCHEMAS['silver']}.users"),
    ("SILVER", f"{SCHEMAS['silver']}.posts"),
    ("SILVER", f"{SCHEMAS['silver']}.comments"),
    ("GOLD",   f"{SCHEMAS['gold']}.posts_per_user"),
    ("GOLD",   f"{SCHEMAS['gold']}.comments_per_post"),
    ("GOLD",   f"{SCHEMAS['gold']}.user_activity_summary"),
]

for layer, table in all_tables:
    cnt = spark.table(table).count()
    print(f"   {layer:7s}  {table:55s}  {cnt:>6} rows")

print("=" * 60)
print("✅ All layers complete! Bronze → Silver → Gold pipeline finished.")

📋 MEDALLION ARCHITECTURE — TABLE INVENTORY
   BRONZE   main.bronze_jsonplaceholder.raw_users                        10 rows
   BRONZE   main.bronze_jsonplaceholder.raw_posts                       100 rows
   BRONZE   main.bronze_jsonplaceholder.raw_comments                    500 rows
   BRONZE   main.bronze_jsonplaceholder.ingestion_audit_log              35 rows
   SILVER   main.silver_jsonplaceholder.users                            10 rows
   SILVER   main.silver_jsonplaceholder.posts                           100 rows
   SILVER   main.silver_jsonplaceholder.comments                        500 rows
   GOLD     main.gold_jsonplaceholder.posts_per_user                     10 rows
   GOLD     main.gold_jsonplaceholder.comments_per_post                 100 rows
   GOLD     main.gold_jsonplaceholder.user_activity_summary              10 rows
✅ All layers complete! Bronze → Silver → Gold pipeline finished.
